# Milestone 6G: End-to-End Big Data AML Architecture Capstone Demo

This capstone notebook orchestrates the complete **Milestone 6 Big Data Enhancement**, executing the integrated pipeline across all six sub-phases:

| Phase | Subsystem | Key Demonstration |
|:---|:---|:---|
| **6A** | Spark RDD Fundamentals | RDD transformations, actions, pair RDDs, exchange rates |
| **6F** | Multi-Source ETL | 3-source ingestion, validation rules, deduplication, enrichment |
| **6E** | Wide-Column Data Model | Delta Lake tables, schema evolution, Cassandra query emulation |
| **6B** | Spark SQL Analytics | Fan-in/out aggregation, hourly velocity windows, mule signatures |
| **6C** | Spark MLlib Classification | Distributed GBT/RF pipelines, PR-AUC evaluation, benchmarks |
| **6D** | Structured Streaming | 5-minute tumbling windows, anomaly rules, real-time alerts |

---


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName('ChainReaction-Capstone')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark Session Ready. Version:', spark.version)


## Phase 1: RDD Ingestion & Pair Transformations (Milestone 6A)


In [ ]:
from src.spark_rdd_pipeline import run_rdd_pipeline
csv_path = '../data/processed/processed_df.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/raw/HI-Small_Trans.csv'

rdd_results = run_rdd_pipeline(spark.sparkContext, csv_path)
print('RDD Phase: Processed', rdd_results['total_accounts'], 'accounts.')
print('Riskiest account:', rdd_results['riskiest'][1], 'Score:', round(rdd_results['riskiest'][0], 4))


## Phase 2: Multi-Source ETL & Data Quality Validation (Milestone 6F)


In [ ]:
from src.etl_pipeline import ETLPipeline
etl = ETLPipeline(
    spark=spark,
    transactions_path=csv_path,
    sanctions_path='../data/external/sanctions_list.csv',
    country_risk_path='../data/external/country_risk_scores.csv',
    max_transaction_rows=25000,
)
etl_result = etl.run()
print('ETL Phase: Quality Audit Summary:')
for k, v in etl.generate_quality_report().items():
    print(f'  {k}: {v}')


## Phase 3: Wide-Column Storage & Schema Evolution (Milestone 6E)


In [ ]:
from src.wide_column_model import WideColumnStore
store = WideColumnStore(spark, base_storage_dir='../outputs/wide_column_store')
store.create_tables(etl_result.validated_transactions, etl_result.enriched_accounts)
queries = store.simulate_cassandra_queries('8000EBD30')
print('Wide-Column Phase: Materialized View Triage:')
queries['high_risk_cluster_triage'].show(5)


## Phase 4: Structured Big Data Analytics with Spark SQL (Milestone 6B)


In [ ]:
from src.spark_sql_analytics import SparkSQLAnalytics
analytics = SparkSQLAnalytics(spark, etl_result.validated_transactions, etl_result.enriched_accounts)
mules_df = analytics.query_pass_through_mules()
print('Spark SQL Phase: Top Pass-Through Mule Signatures:')
mules_df.show(5)


## Phase 5: Distributed Machine Learning with MLlib (Milestone 6C)


In [ ]:
from src.spark_ml_pipeline import SparkMLMuleClassifier
features_path = '../data/processed/account_features.csv'
split_path = '../data/processed/split_accounts.csv'

raw_features = spark.read.option('header', 'true').option('inferSchema', 'true').csv(features_path)
split_df = spark.read.option('header', 'true').csv(split_path)

classifier = SparkMLMuleClassifier(spark)
train_df, val_df, test_df = classifier.prepare_data(raw_features, split_df)

# Fast evaluation on samples
s_train = train_df.sample(False, 0.15, seed=42)
s_val = val_df.sample(False, 0.25, seed=42)
s_test = test_df.sample(False, 0.25, seed=42)

ml_results = classifier.run_benchmark_suite(s_train, s_val, s_test)
classifier.print_comparison_table(ml_results)


## Phase 6: Real-Time Structured Streaming (Milestone 6D)


In [ ]:
from src.streaming_pipeline import StreamingAnomalyDetector
from src.stream_simulator import simulate_stream

detector = StreamingAnomalyDetector(spark)
simulate_stream(csv_path, detector.incoming_dir, batch_size=100, delay_seconds=0.1, max_batches=2)
queries = detector.start_pipeline(enable_console=True, enable_parquet=False, trigger_interval='1 second', timeout_seconds=5)
print('Streaming Phase Completed successfully.')


## 7. Comprehensive Requirements Traceability Matrix

| Requirement | Pipeline Deliverable | Verified In Notebook |
|:---|:---|:---|
| **Spark RDD Transformations & Actions** | `src/spark_rdd_pipeline.py` | Cell 2 (RDD Phase) & `06a_spark_rdd_fundamentals.ipynb` |
| **Multi-Source ETL, Mapping & Validation** | `src/etl_pipeline.py` | Cell 3 (ETL Phase) & `06f_etl_pipeline.ipynb` |
| **Wide-Column Data Model & Schema Evolution** | `src/wide_column_model.py`, `docs/cassandra_schema.cql` | Cell 4 (Wide-Column Phase) & `06e_wide_column_model.ipynb` |
| **Spark SQL Structured Analytics** | `src/spark_sql_analytics.py` | Cell 5 (Spark SQL Phase) & `06b_spark_sql_analytics.ipynb` |
| **Spark MLlib Classification & Benchmarking** | `src/spark_ml_pipeline.py` | Cell 6 (MLlib Phase) & `06c_spark_ml_pipeline.ipynb` |
| **Real-Time Structured Streaming** | `src/streaming_pipeline.py`, `src/stream_simulator.py` | Cell 7 (Streaming Phase) & `06d_streaming_pipeline.ipynb` |
| **Databricks Community Edition Readiness** | `notebooks/06*.ipynb` | Micro-cluster compatible execution flow |

---
### Capstone Execution Concluded.


In [ ]:
spark.stop()
print('Spark session stopped. Capstone demonstration complete!')
